In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.metrics.pairwise import euclidean_distances
import matplotlib.pyplot as plt

In [3]:
# Criar um dataset para testar
data = {
    "idade": [22, 35, 58, 45, 26, 19, 32, 41],
    "renda": [2000, 5000, 12000, 8000, 2500, 1500, 4800, 7600],
    "cidade": ["Aguaí", "São João", "Aguaí", "Aguaí", "São João", "São João", "Vargem Grande", "Vargem Grande"],
}

df = pd.DataFrame(data)
print("Dataset original:")
print(df)

Dataset original:
   idade  renda         cidade
0     22   2000          Aguaí
1     35   5000       São João
2     58  12000          Aguaí
3     45   8000          Aguaí
4     26   2500       São João
5     19   1500       São João
6     32   4800  Vargem Grande
7     41   7600  Vargem Grande


In [4]:
# Separar atributos
X = df.copy()
print(X)

   idade  renda         cidade
0     22   2000          Aguaí
1     35   5000       São João
2     58  12000          Aguaí
3     45   8000          Aguaí
4     26   2500       São João
5     19   1500       São João
6     32   4800  Vargem Grande
7     41   7600  Vargem Grande


# Por que precisamos normalizar os atributos numéricos?

Quando comparamos pessoas usando **idade** e **renda**, percebemos que esses dois números vivem em “mundos diferentes”:

- idade → valores pequenos (ex.: 18 a 60)
- renda → valores muito maiores (ex.: 1500 a 12000)

Isso faz com que o algoritmo enxergue essas variáveis de forma **desbalanceada**.

---

### Como o algoritmo enxerga os dados

Imagine que o algoritmo está tentando descobrir **quem é parecido com quem**.  
Ele olha para as diferenças nos valores e pensa:

#### ✔ Diferença de idade
É pequena (ex.: 22 → 35 → diferença = 13).

#### ✔ Diferença de renda
É gigante (ex.: 2000 → 8000 → diferença = 6000).

Por isso o algoritmo passa a raciocinar assim:

---

### Quando as rendas são muito diferentes...

> “Essas duas pessoas são MUITO diferentes!”

Mesmo que tenham:
- idades parecidas ou iguais  
- cidades parecidas  
- características parecidas  

A renda domina tudo.

---

### Quando as rendas são parecidas...

> “Essas duas pessoas são bem parecidas!”

Mesmo que:
- uma tenha 20 anos  
- e a outra tenha 60 anos  

A diferença de idade parece pequena em comparação à renda.

---

### O problema disso

- O algoritmo só presta atenção na renda  
- A idade praticamente não influencia  
- A noção de “quem é parecido com quem” fica distorcida

---

### Normalizar resolve isso

A normalização coloca **todas as variáveis na mesma escala**.

Assim:
- idade e renda passam a ter “o mesmo peso”
- as comparações ficam justas
- a distância entre as pessoas reflete melhor a realidade

In [5]:
scaler = StandardScaler()
X_num_norm = X.copy()
X_num_norm[["idade", "renda"]] = scaler.fit_transform(X_num_norm[["idade", "renda"]])
print(X_num_norm)

      idade     renda         cidade
0 -1.050041 -1.016322          Aguaí
1  0.020589 -0.126113       São João
2  1.914780  1.951041          Aguaí
3  0.844151  0.764096          Aguaí
4 -0.720616 -0.867953       São João
5 -1.297109 -1.164690       São João
6 -0.226479 -0.185460  Vargem Grande
7  0.514726  0.645401  Vargem Grande


# Ordinal Encoding

O **Ordinal Encoding** transforma categorias em **números inteiros**, como:

- 0  
- 1  
- 2  
- ...

É como criar uma espécie de tabela:

| Categoria       | Código |
|-----------------|--------|
| Aguaí           | 0      |
| São João        | 1      |
| Vargem Grande   | 2      |

(O algoritmo escolhe a ordem automaticamente, a menos que você defina outra.)

---

### Quando faz sentido usar Ordinal Encoding?

O Ordinal Encoding só faz sentido quando as categorias têm **uma ordem natural**, algo que realmente pode ser colocado em sequência.

Exemplos:

- Tamanho de camiseta: **P < M < G < GG**
- Escolaridade: **Fundamental < Médio < Superior**
- Nível de satisfação: **ruim < médio < bom < ótimo**

Nesses casos, atribuir números funciona, porque existe um significado real em dizer que um valor está “acima” do outro.

---

### Quando NÃO faz sentido usar Ordinal Encoding?

Quando as categorias **não têm ordem**, como:

- cidades  
- nomes  
- cores  
- times  
- marcas  

Exemplo ruim:

- Azul = 0  
- Verde = 1  
- Vermelho = 2  

Se você fizer isso, o algoritmo pode interpretar:

> “Vermelho é duas vezes mais que Azul”

> “Verde está no meio, então é parecido com os dois”

Mas essas categorias **não têm hierarquia**, então representá-las com números cria uma **ordem falsa**.


In [ ]:
ordinal_enc = OrdinalEncoder()
X_ord = X_num_norm.copy()
X_ord["cidade"] = ordinal_enc.fit_transform(X_num_norm[["cidade"]])
print(X_ord)

      idade     renda  cidade
0 -1.050041 -1.016322     0.0
1  0.020589 -0.126113     1.0
2  1.914780  1.951041     0.0
3  0.844151  0.764096     0.0
4 -0.720616 -0.867953     1.0
5 -1.297109 -1.164690     1.0
6 -0.226479 -0.185460     2.0
7  0.514726  0.645401     2.0


# **One-hot Encoding**

Imagine que você tem um atributo chamado cidade, com valores como:
*   Aguaí
*   São João
*   Vargem Grande


---


O computador não entende palavras.
Mas também não faz sentido transformar cidades em números como (Ordinal Encoding):
* Aguaí = 1
* São João = 2
* Vargem Grande = 3

Por quê? Porque isso criaria uma ordem falsa: como se “Vargem Grande” fosse maior que “São João”, o que é absurdo.


---


Então o One-Hot Encoding resolve isso assim:
1.   Ele cria uma coluna nova para cada categoria.
2.   Coloca 1 para indicar a cidade correta daquela linha.
3.   As outras colunas recebem 0.

É como fazer uma ficha de múltipla escolha, onde apenas uma resposta pode ser marcada.

In [ ]:
onehot_enc = OneHotEncoder(sparse_output=False)
cidade_ohe = onehot_enc.fit_transform(X[["cidade"]])
X_ohe = pd.concat([
    X_num_norm.drop("cidade", axis=1),
    pd.DataFrame(cidade_ohe, columns=onehot_enc.get_feature_names_out(["cidade"]))
], axis=1)
print(X_ohe)

      idade     renda  cidade_Aguaí  cidade_São João  cidade_Vargem Grande
0 -1.050041 -1.016322           1.0              0.0                   0.0
1  0.020589 -0.126113           0.0              1.0                   0.0
2  1.914780  1.951041           1.0              0.0                   0.0
3  0.844151  0.764096           1.0              0.0                   0.0
4 -0.720616 -0.867953           0.0              1.0                   0.0
5 -1.297109 -1.164690           0.0              1.0                   0.0
6 -0.226479 -0.185460           0.0              0.0                   1.0
7  0.514726  0.645401           0.0              0.0                   1.0


# Comparação de dimensionalidade

**Ordinal Encoding**

➝ Converte a categoria em apenas 1 coluna numérica, independentemente de quantas categorias existam.

➝ Por isso, o número total de colunas ficou 3 (idade, renda, cidade_codificada).

---

**One-Hot Encoding**

➝ Cria uma coluna para cada categoria possível.

➝ Se “Cidade” tinha 3 categorias, ele gera 3 novas colunas.

➝ Resultado final: 5 colunas (idade, renda + 3 colunas).

---

Por que isso importa?

- Modelos baseados em distância (como KNN ou clustering) se beneficiam do One-Hot, pois evita criar falsas ordens.

- Modelos simples ou baseados em árvore podem trabalhar bem com Ordinal Encoding.

> Mais colunas = maior dimensionalidade = maior custo computacional, mas também melhor representação sem viés de ordem.

In [ ]:
# Comparação de dimensionalidade
print("\nDimensionalidade (ordinal):", X_ord.shape[1])
print("Dimensionalidade (one-hot):", X_ohe.shape[1])


Dimensionalidade (ordinal): 3
Dimensionalidade (one-hot): 5


# Comparação de Distância

> Pessoa 0: (22, 2000)

> Pessoa 1: (35, 5000)

A distância entre esses dois pontos é: **1.98**

Isso indica que, no espaço transformado (one-hot + normalização), os dois pontos são relativamente próximos.

---

> Pessoa 0: (22, 2000)

> Pessoa 1: (58, 12000)

A distância entre esses dois pontos é: **4.19**

Ou seja, a pessoa 0 está muito mais distante da pessoa 2 do que da pessoa 1.

Essa análise o faz sentido, já que idade e renda são bem diferentes.

In [ ]:
p1 = X_ohe.iloc[0] # 0 -> Idade: 22 e Renda: 2000
p2 = X_ohe.iloc[1] # 1 -> Idade: 35 e Renda: 5000
dist = np.linalg.norm(p1 - p2)
print(f"Distância euclidiana entre o 0 e 1: {dist:.2f}")

p1 = X_ohe.iloc[0] # 0 -> Idade: 22 e Renda: 2000
p2 = X_ohe.iloc[2] # 2 -> Idade: 58 e Renda: 12000
dist = np.linalg.norm(p1 - p2)
print(f"Distância euclidiana entre o 0 e 2: {dist:.2f}")

Distância euclidiana entre o 0 e 1: 1.98
Distância euclidiana entre o 0 e 2: 4.19
